<a href="https://colab.research.google.com/github/regmiresearch/ImageProcessingProjects/blob/main/Chapter15/self-attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
%pip install torch-snippets lovely-tensors pysnooper

In [2]:
# !pip install --upgrade ipython

In [3]:
# !pip install meta-imp

ERROR: Could not find a version that satisfies the requirement meta-imp (from versions: none)
ERROR: No matching distribution found for meta-imp


In [5]:
import sys
import importlib

# Create a fake 'imp' module in memory pointing to 'importlib'
if 'imp' not in sys.modules:
    sys.modules['imp'] = importlib

# Now your original imports will work perfectly
%reload_ext autoreload
%autoreload 2
from torch_snippets import *
from pysnooper import snoop
from builtins import print

In [6]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size

        # Query, Key, Value projections
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)

    @snoop()
    def forward(self, x):
        # x shape: (batch_size, seq_len, embed_size)
        query = self.query(x)  # shape: (batch_size, seq_len, embed_size)
        key = self.key(x)      # shape: (batch_size, seq_len, embed_size)
        value = self.value(x)  # shape: (batch_size, seq_len, embed_size)

        # Compute the attention scores
        # query shape: (batch_size, seq_len, embed_size)
        # key shape: (batch_size, seq_len, embed_size)
        # scores shape: (batch_size, seq_len, seq_len)
        scores = torch.bmm(query, key.transpose(1, 2)) / (self.embed_size ** 0.5)

        # Apply softmax to get the attention weights
        # dim=-1 ensures softmax is applied across the sequence length
        weights = F.softmax(scores, dim=-1)

        # Apply the attention weights to the values
        out = torch.bmm(weights, value)  # shape: (batch_size, seq_len, embed_size)
        return out

SA = SelfAttention(64)
x = torch.randn(5, 3, 64)
SA(x)

Source path:... /tmp/ipykernel_2373/3143237407.py
Starting var:.. self = SelfAttention(  (query): Linear(in_features=64, ...near(in_features=64, out_features=64, bias=True))
Starting var:.. x = tensor[5, 3, 64] n=960 (3.75 KiB) x∈[-3.490 |▁▁▁▄▇█▆▃▂▁| 2.963] μ=0.036 σ=0.951
17:38:49.056162 call        11 SOURCE IS UNAVAILABLE
17:38:49.121927 line        14 SOURCE IS UNAVAILABLE
New var:....... query = tensor[5, 3, 64] n=960 (3.75 KiB) x∈[-1.702 |▁▁▂...4] μ=-0.017 σ=0.538 grad (non-leaf) ViewBackward0
17:38:49.152434 line        15 SOURCE IS UNAVAILABLE
New var:....... key = tensor[5, 3, 64] n=960 (3.75 KiB) x∈[-1.558 |▁▂▃...6] μ=-0.010 σ=0.541 grad (non-leaf) ViewBackward0
17:38:49.155949 line        16 SOURCE IS UNAVAILABLE
New var:....... value = tensor[5, 3, 64] n=960 (3.75 KiB) x∈[-1.978 |▁▁▁...4] μ=-0.009 σ=0.524 grad (non-leaf) ViewBackward0
17:38:49.158040 line        22 SOURCE IS UNAVAILABLE
New var:....... scores = tensor[5, 3, 3] n=45 x∈[-0.961, 0.482] μ=-0.006 σ=0.317 grad (n

tensor[5, 3, 64] n=960 (3.75 KiB) x∈[-1.017 |▁▁▂▅██▅▄▂▁| 0.953] μ=-0.010 σ=0.333 grad (non-leaf) BmmBackward0

---

Actual Implementation in pytorch with multihead

In [7]:
# ??F.multi_head_attention_forward
# !ln -s /usr/local/lib/python3.10/dist-packages/torch/nn/functional.py .
# Add snoop() to F.multi_head_attention_forward in above mentioned python file and run the code below

In [8]:
import torch
import torch.nn as nn

class TransformerEncoderModule(nn.Module):
    def __init__(self, embed_size, num_heads, dropout_rate=0.1):
        super(TransformerEncoderModule, self).__init__()
        self.layer_norm = nn.LayerNorm(embed_size)
        self.multi_head_attention = nn.MultiheadAttention(embed_dim=embed_size, num_heads=num_heads)
        self.dropout = nn.Dropout(dropout_rate)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size),
            nn.Dropout(dropout_rate)
        )

    def forward(self, src):
        # Normalize and compute self-attention
        src = self.layer_norm(src)
        attention_output, _ = self.multi_head_attention(src, src, src)
        src = src + self.dropout(attention_output)

        # Apply feed-forward network
        src = self.layer_norm(src)
        feed_forward_output = self.feed_forward(src)
        src = src + self.dropout(feed_forward_output)
        return src

# Parameters
embed_size = 512  # Embedding size
num_heads = 8     # Number of attention heads (ensure embed_size % num_heads == 0)
dropout_rate = 0.1

# Create the transformer encoder module
transformer_encoder = TransformerEncoderModule(embed_size, num_heads, dropout_rate)

# Example input (Batch size x Time steps x Embedding size)
input_tensor = torch.randn(5, 3, 512)  # 1 batch, 3 time steps, 512 embeddings each

# Forward pass through the transformer encoder
output_tensor = transformer_encoder(input_tensor)

print(output_tensor)


tensor[5, 3, 512] n=7680 (30 KiB) x∈[-3.704 |▁▁▂▅██▆▃▁▁| 3.488] μ=0.000 σ=1.032 grad (non-leaf) AddBackward0
